In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:


import importlib
import ewxndfd.ndfd_forecast_api
importlib.reload(ewxndfd)
from ewxndfd import ndfd_forecast_api as ndfd
print(ndfd.PKG_VERSION)


202606261745


In [3]:

lat, lon = ndfd.LANSING_LAT_LON
 
# currently broken
#daily_forecast_df = ndfd.daily_forecast_summary(lat=lat, lon = lon, 
#                                                hourly_weather=None, 
#                                                units = 'US')
#daily_forecast_df

### Examine temperatures in detail

Temps are not forecasted for today depending on what time it is.  Meteorologists assume the min temp will happen prior to Noon, and if it's already happened, it's not a forecast.   (this is an incorrect assumptions, there are many days when the min temp happens after noon)

In [4]:
# get a response
units = "US"  # use us units to compare with NWS forecast
resp = ndfd.request_ndfd_digital_forecast(lat=lat, lon=lon, 
                                          user_agent = ndfd.DEFAULT_USER_AGENT, 
                                          units=units)


optionally print whole xml

In [ ]:
# print(resp.text)

look at XML for temperatures

In [5]:
root = ndfd.ET.fromstring(resp.text)

metric_path = ".//temperature[@type='minimum']"
metric_name = ndfd.weather_metric_name_from_xml(root, metric_path)
min_temperature_df = ndfd.weather_metric_xml_to_df(root, metric_path)
min_temperature_df

using end times


,end_time,value,date
0,2026-06-27T09:00:00-04:00,57,2026-06-27
1,2026-06-28T09:00:00-04:00,57,2026-06-28
2,2026-06-29T09:00:00-04:00,66,2026-06-29
3,2026-06-30T09:00:00-04:00,73,2026-06-30
4,2026-07-01T09:00:00-04:00,75,2026-07-01
5,2026-07-02T09:00:00-04:00,73,2026-07-02
6,<NA>,<NA>,2026-06-26


In [ ]:
import pandas as pd
from datetime import date

df = min_temperature_df
min_forecast_date = min(df['forecast_date'])
if(not (min_forecast_date)==date.today()):
    new_row = {metric_name : None, 'forecast_date': date.today()}
    df.loc[len(df)] = new_row

pd.DataFrame.convert_dtypes(df)

In [ ]:
metric_path = ".//temperature[@type='maximum']"
metric_name = ndfd.weather_metric_name_from_xml(root, metric_path)
max_temperature_df = ndfd.weather_metric_xml_to_df(root, metric_path)
max_temperature_df


In [ ]:
min(max_temperature_df['forecast_date']) ==date.today()

One min temp forecast per day before X o'clock
- at 12pm 26June, times are 8pm today to 9 am tomorrow.  
```
<start-valid-time>2026-06-26T20:00:00-04:00</start-valid-time>
<end-valid-time>2026-06-27T09:00:00-04:00</end-valid-time>
```
- In the morning, there is a min temperature for today  
- after morning there is no min temp forecast for today, so the first day is tomorrow. 
- times list are 0900 to 0400 which must be incorrect

Mapping of NDFD min temp forecast" to our "daily forecast" notion. 

00 < Time < 0900 : 
- date of min temp start time is yesterday
- date of min temp end-valid time is today
- date = today  ( NOT min(date(startime)) ) how to tell use end or start time???
- forecast = value

0900 < Time < 2000 :
-date of minT start time is today BUT THAT'S TOMORROW FORECAST
-date if minT end time is tomorrow
-date = today <- how do we get this from ?
-forecast = Null

2000 < t < 2359
- -date of minT start time is 

considering how to determine if should use NULL or value for forecast for today.   If the 2000 < time, 
(e.g. it's after 8pm) the min. date for start date is today.  There will always be a forecast for today from min 

use start time for forecast
I don't think NDFD has the notion of "daily min temp" forecast for a date since the time period is over
two different days and that min temp could happen at any time during the period 8pm Day 0 to 9pm Day 1. 

   

In [ ]:
metric_path = ".//temperature[@type='minimum']"
metric_name = ndfd.weather_metric_name_from_xml(root, metric_path)
weather_values = root.find(metric_path)
print(ndfd.ET.tostring(weather_values, encoding='unicode'))


In [ ]:
time_layout_key = weather_values.get('time-layout')
time_layouts = root.findall('.//time-layout')
time_layouts
for tl in time_layouts:
    layout_key = tl.find('layout-key').text
    if layout_key == time_layout_key:
        s, e = ndfd.get_start_end_times(root, time_layout_key)
        print(ndfd.ET.tostring(tl, encoding='unicode'))
        print(s,e)
        break
    



In [ ]:
for tl in time_layouts:
    layout_key = tl.find('layout-key').text
    print(f"found key {layout_key}")
    print(f"our key {time_layout_key}")
    if layout_key == time_layout_key:
        end_times = [st.text for st in tl.findall('end-valid-time')]
        break

end_times

assumption from NDFD is that 
    low temps occur between day-1 20:00 day 0 09:00
    if current time > 9:00 and current time < 20:00: no min temp forecast for today (only) = None (as it's already occurred), use min of min hourly temp for today 
    if current time > = 20:00 current_time <= 9:00: 

alternative strategy: 
    data frame dates = date( start_time)'s 
    if data frame does NOT have a row with date == today:
        add blank row with date = today and value (min temp/max temp) = None/Null

- when time(now) > 09:00 (min temp is already known), will add row with value = None/Null
- when time(now) > 20:00 (max temp is already known), will add row with value = None/Null



